# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

def scrape_books(min_rating, max_price):
    base_url = "https://books.toscrape.com/"
    page_url = urljoin(base_url, "catalogue/page-1.html")

    books_data = []

    # Convert rating words to numbers
    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    while page_url:
        response = requests.get(page_url)
        soup = BeautifulSoup(response.content, "html.parser")

        books = soup.find_all("article", class_="product_pod")

        for book in books:
            # Title
            title = book.h3.a["title"]

            # Price
            price_text = book.find("p", class_="price_color").text.strip()
            price = float(price_text.replace("£", "").replace("Â", ""))

            # Rating
            rating_class = book.find("p", class_="star-rating")["class"]
            rating_word = rating_class[1]
            rating = rating_map[rating_word]

            # Availability
            availability = book.find("p", class_="instock availability").text.strip()

            # Only continue if book passes filters
            if rating >= min_rating and price <= max_price:
                # Build book detail URL
                relative_url = book.h3.a["href"]
                book_url = urljoin(page_url, relative_url)

                # Request detail page
                detail_response = requests.get(book_url)
                detail_soup = BeautifulSoup(detail_response.content, "html.parser")

                # UPC
                table = detail_soup.find("table", class_="table table-striped")
                upc = table.find_all("tr")[0].find("td").text.strip()

                # Genre from breadcrumb
                breadcrumb = detail_soup.find("ul", class_="breadcrumb").find_all("li")
                genre = breadcrumb[2].text.strip()

                # Description
                description = "No description available"
                description_header = detail_soup.find("div", id="product_description")
                if description_header:
                    description = description_header.find_next("p").text.strip()

                # Save row
                books_data.append({
                    "UPC": upc,
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating,
                    "Genre": genre,
                    "Availability": availability,
                    "Description": description
                })

        # Go to next page if it exists
        next_button = soup.find("li", class_="next")
        if next_button:
            next_page = next_button.find("a")["href"]
            page_url = urljoin(page_url, next_page)
        else:
            page_url = None

    return pd.DataFrame(books_data)

In [8]:
books_df = scrape_books(min_rating=4, max_price=20)
books_df.head()

KeyboardInterrupt: 

In [7]:
books_df.to_csv("books_to_scrape_filtered.csv", index=False)

NameError: name 'books_df' is not defined